# Human-in-the-Loop: Brány pred akciou, triedenie rizika a auditové záznamy

README pre túto lekciu predstavuje Human-in-the-Loop s krátkym útržkom, ktorý požiada používateľa o `APPROVE` alebo `REJECT` po tom, čo agent už vytvoril odpoveď. Tento vzor je dobrým východiskom, ale produkčné implementácie HITL zvyčajne potrebujú tri ďalšie časti:

1. **bránu pred akciou**, ktorá beží **predtým**, než agent vykoná rizikový krok, aby sa udržali náklady, nezvratnosť a latencia pod kontrolou.
2. **triedenie rizika**, takže nízkorizikové akcie sa vykonávajú automaticky, stredne rizikové akcie sa schvaľujú hromadne a iba vysoko rizikové akcie blokujú činnosť človeka.
3. **auditový záznam plus slučka revízie**, takže každé rozhodnutie brány je zaznamenané ako JSONL a zamietnutie znovu vyzve agenta so štruktúrovaným dôvodom namiesto jednoduchého vytlačenia `Revising...`.

Tento notebook stavia každú z týchto častí na tých istých primitívach ako `06-system-message-framework.ipynb`. Beží end-to-end v režime `DEMO_MODE = True` (nie je potrebný interaktívny vstup) alebo s reálnymi výzvami `input()` keď je `DEMO_MODE = False`. Poznámka: v DEMO_MODE je opakovanie tretej cieľovej časti naprogramované, takže mechanika slučky je viditeľná end-to-end. Revíziou riadená preklasifikácia vyžaduje `DEMO_MODE = False` a operátora.

**Mimo rozsahu (riešené v iných lekciách):** autentifikácia a kontrola prístupu (Lesson 06 README threat 2), middleware na volanie nástrojov (Lesson 14 MAF deep dive), vzorce viacagentových debát.

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

token = os.environ.get("GITHUB_TOKEN", "")
if not token:
    raise RuntimeError(
        "GITHUB_TOKEN environment variable is not set. This notebook needs "
        "a GitHub PAT with 'Models: Read-only' permission. Set GITHUB_TOKEN "
        "in your environment or a local .env file before running."
    )

endpoint = "https://models.github.ai/inference"
model_name = "gpt-4o"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


## Vzor 1: Brána pred akciou

Úryvok HITL v README najprv zavolá agentovi, potom požiada používateľa o schválenie výstupu. To je **priebeh po akcii**. Agent už vykonal činnosť, čiže náklady na volanie LLM sú už zaplatené a akýkoľvek vedľajší efekt (odoslaný email, zapísaný riadok v databáze, pridaný komentár) už nastal.

**Priebeh pred akciou** vkladá bránu pred spustením rizikového kroku agentom. Agent navrhne akciu, brána rozhodne, či sa vykoná, a vedľajší efekt nastane len po schválení.

| Aspekt | Schválenie po akcii (úryvok README) | Brána pred akciou (tento notebook) |
|---|---|---|
| Kedy prebieha schválenie? | Po tom, čo agent vygeneroval výstup | Pred vykonaním akéhokoľvek vedľajšieho efektu |
| Náklady na LLM pri odmietnutí | Už zaplatené | Zaplatené len za návrh, nie za akciu |
| Nezvratné vedľajšie efekty | Možné (akcia sa už vykonala) | Zamedzené |
| Jasnosť auditu | Schválenie je výpisová správa | Schválenie je JSON záznam s časovou pečiatkou, akciou, dôvodom |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Vzor 2: Rizikové vrstvenie

Nie každá akcia potrebuje schválenie človekom. Prehliadanie vo verejnom API s právami len na čítanie má iné riziká ako odoslanie emailu zákazníkovi. Zaobchádzanie s oboma rovnako plytvá pozornosťou operátora a spomaľuje agenta.

Jednoduchý model so 3 úrovňami:

| Vrstva | Príklady | Schvaľovací proces |
|---|---|---|
| `nízka` (len na čítanie) | Vyhľadávanie v databáze znalostí, hľadanie možností letu, načítanie verejnej webovej stránky | Automatické vykonanie, zaznamenané na audit |
| `stredná` (lacná mutácia) | Uloženie výsledku do vyrovnávacej pamäte, zvýšenie čítača, naplánovanie pripomienky | Automatické vykonanie, ale prehľad zoskupený denne |
| `vysoká` (externé alebo nezvratné) | Odoslanie emailu, zaúčtovanie platby, príspevok do verejného kanála | Zastavenie na schválenie človekom |

Toto je jedno vrstvenie. Produkčné systémy často používajú podrobnejšie vrstvy (napr. úrovne povolení AWS IAM, vrstvy prístupu založené na rolách). Trojvrstvová verzia nižšie je najmenšou užitočnou verziou pre agenta, ktorý kombinuje akcie len na čítanie a akcie vyvolávajúce vedľajšie efekty.

Klasifikátor nižšie používa heuristiky založené na kľúčových slovách, aby demo zostalo deterministické a lacné. V produkčnom systéme by ste to nahradili naučeným klasifikátorom alebo politickým motorom.

In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Vzor 3: Auditný záznam a revízna slučka

`print("Response approved.")` nie je auditný záznam. Pre dôveru by malo byť každé rozhodnutie brány zaznamenané ako štruktúrovaná udalosť, ktorú môžete neskôr dotazovať, prehrávať alebo pripojiť k prehliadke incidentu.

Dva prvky:

1. **Len na pridávanie JSONL.** Jeden riadok na rozhodnutie, s časovou značkou, akciou, úrovňou, rozhodnutím, dôvodom. Ľahko sa dá grepovať, ľahko sa neskôr odosiela do skutočného úložiska záznamov.
2. **Revízna slučka pri zamietnutí.** Keď brána vráti `deny`, agent sám sebe znova položí otázku s dôvodom zamietnutia v kontexte, takže ďalší návrh sa môže vyhnúť problému.

In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.complete(
        model=model_name,
        messages=[SystemMessage(content=system), UserMessage(content=user_text)],
    )
    return response.choices[0].message.content.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}


In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Dodatočné zdroje

Niekoľko ďalších verejných projektov implementuje varianty týchto vzorov HITL. Porovnajte prístupy, aby ste našli ten, ktorý vyhovuje vášmu stacku:

- **LangChain** obaly nástrojov s človekom v slučke ([dokumentácia](https://python.langchain.com/docs/integrations/tools/human_tools)): obaly nástrojov, ktoré pozastavia vykonávanie pre vstup od človeka.
- **AutoGen** `UserProxyAgent` ([dokumentácia v0.2](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ to preštruktúroval): používa rolu agenta špeciálne na reprezentáciu človeka v konverzáciách viacerých agentov.
- **Semantic Kernel** filtre funkcií ([dokumentácia](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/filters)): middleware, ktorý beží okolo každého volania funkcie, vhodný pre bránenie prístupu.

Každý projekt spracováva tri podvzory inak: LangChain ich zabalí ako nástroje, AutoGen používa rolu agenta, Semantic Kernel používa middleware filtre. Prečítajte si jednu alebo dve implementácie od začiatku do konca predtým, než si vyberiete dizajn pre svojho vlastného agenta.

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vyhlásenie o zodpovednosti**:
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Hoci sa snažíme o presnosť, vezmite prosím na vedomie, že automatické preklady môžu obsahovať chyby alebo nepresnosti. Pôvodný dokument v jeho natívnom jazyku by mal byť považovaný za autoritatívny zdroj. Pre kritické informácie sa odporúča profesionálny ľudský preklad. Nie sme zodpovední za žiadne nedorozumenia alebo nesprávne interpretácie vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
